# FW-LNSA Manuscript Readiness Execution

This notebook completes the final pre-writing workflow:

1. Mount persistent Google Drive storage.
2. Clone the exact private repository revision.
3. Validate the complete implementation.
4. Verify the completed FW-LNSA confirmatory artifacts.
5. Run hash-compatible validation-calibrated baselines.
6. Generate paired statistics, calibration analysis, publication tables, and figures.
7. Run the final readiness gate.
8. Build the focused writer handoff package for Tanazzah.

The final test partitions are never used for feature selection, model selection, or threshold calibration. The workflow is resumable. Do not delete `results_manuscript/` while a run is in progress.


## Required Google Drive layout

```text
MyDrive/FW-LNSA-NIDS/
├── data/
│   ├── nsl_kdd/
│   │   ├── KDDTrain+.txt
│   │   └── KDDTest+.txt
│   └── cicids2017/
│       └── MachineLearningCSV.zip
├── results_confirmatory/
│   ├── nsl_kdd/
│   └── cicids2017/
└── 08_manuscript_readiness_execution.ipynb
```

The private repository requires a Colab secret named `GITHUB_TOKEN` with **Contents: Read-only** permission for `sarosh-jawed/FW-LNSA-NIDS`.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Configure paths and verify required inputs


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/FW-LNSA-NIDS")
DATA_ROOT = DRIVE_ROOT / "data"
NSL_TRAIN = DATA_ROOT / "nsl_kdd" / "KDDTrain+.txt"
NSL_TEST = DATA_ROOT / "nsl_kdd" / "KDDTest+.txt"
CIC_ARCHIVE = DATA_ROOT / "cicids2017" / "MachineLearningCSV.zip"
CONFIRMATORY_ROOT = DRIVE_ROOT / "results_confirmatory"
SMOKE_ROOT = DRIVE_ROOT / "results_manuscript_smoke"
MANUSCRIPT_ROOT = DRIVE_ROOT / "results_manuscript"
HANDOFF_ROOT = MANUSCRIPT_ROOT / "writer_handoff"
NOTEBOOK_COPY = DRIVE_ROOT / "08_manuscript_readiness_execution.ipynb"
REPO_DIR = Path("/content/FW-LNSA-NIDS")
REPO_URL = "https://github.com/sarosh-jawed/FW-LNSA-NIDS.git"

required = [
    NSL_TRAIN,
    NSL_TEST,
    CIC_ARCHIVE,
    CONFIRMATORY_ROOT / "nsl_kdd" / "manifests" / "execution_manifest.json",
    CONFIRMATORY_ROOT / "cicids2017" / "manifests" / "execution_manifest.json",
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required Drive inputs\n" + "\n".join(map(str, missing)))

for directory in [SMOKE_ROOT, MANUSCRIPT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

print("All required datasets and confirmatory artifacts are present.")
print("Persistent manuscript result root:", MANUSCRIPT_ROOT)


## 3. Clone or update the private repository


In [ ]:
import os
import subprocess
from google.colab import userdata

TOKEN = userdata.get("GITHUB_TOKEN")
if not TOKEN:
    raise RuntimeError("GITHUB_TOKEN is missing from the Colab Secrets panel.")

auth_url = REPO_URL.replace("https://", f"https://x-access-token:{TOKEN}@")
env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "remote", "set-url", "origin", auth_url], cwd=REPO_DIR, check=True, env=env)
    subprocess.run(["git", "fetch", "--prune", "origin"], cwd=REPO_DIR, check=True, env=env)
    subprocess.run(["git", "checkout", "main"], cwd=REPO_DIR, check=True, env=env)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR, check=True, env=env)
else:
    subprocess.run(["git", "clone", auth_url, str(REPO_DIR)], check=True, env=env)

subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR, check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True
).stdout.strip()
status = subprocess.run(
    ["git", "status", "--short"], cwd=REPO_DIR, check=True, capture_output=True, text=True
).stdout.strip()
print("Repository commit:", commit)
print("Working tree:", "clean" if not status else status)


## 4. Install the research environment

The locked file preserves the core versions recorded by the completed confirmatory manifests. If Colab reports that a runtime restart is required, restart the session once and rerun Sections 1 through 4 before continuing.


In [ ]:
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-research-lock.txt"],
    cwd=REPO_DIR,
    check=True,
)

import numpy, pandas, sklearn, scipy, yaml, psutil
versions = {
    "numpy": numpy.__version__,
    "pandas": pandas.__version__,
    "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "PyYAML": yaml.__version__,
    "psutil": psutil.__version__,
}
print(versions)
required_versions = {
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "scipy": "1.16.3",
    "PyYAML": "6.0.3",
    "psutil": "5.9.5",
}
if versions != required_versions:
    raise RuntimeError(
        "The active Python process is not using the locked research versions. "
        "Restart the Colab session, then rerun Sections 1 through 4."
    )
print("Locked research environment verified.")


## 5. Validate every implementation suite


In [ ]:
subprocess.run(
    [sys.executable, "scripts/validate_manuscript_readiness.py", "--code-only"],
    cwd=REPO_DIR,
    check=True,
)


## 6. Run a reduced baseline smoke check

This writes only to `results_manuscript_smoke/`. It verifies the exact partition and selected-feature locks before the longer final run.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/run_confirmatory_baselines.py",
        "--config", "configs/manuscript_readiness.yaml",
        "--profile", "smoke",
        "--dataset", "nsl_kdd",
        "--confirmatory-results-dir", str(CONFIRMATORY_ROOT),
        "--output-dir", str(SMOKE_ROOT),
        "--nsl-train", str(NSL_TRAIN),
        "--nsl-test", str(NSL_TEST),
        "--cic-raw-dir", str(CIC_ARCHIVE.parent),
        "--cic-archive", str(CIC_ARCHIVE),
    ],
    cwd=REPO_DIR,
    check=True,
)


In [ ]:
import pandas as pd

smoke_table = pd.read_csv(
    SMOKE_ROOT / "confirmatory_baselines" / "nsl_kdd" / "tables" / "baseline_seed_results.csv"
)
assert len(smoke_table) == 4, f"Expected 4 smoke rows, found {len(smoke_table)}"
assert not smoke_table["test_metrics_used_for_selection"].astype(bool).any()
display(smoke_table[[
    "model_name", "feature_set", "target_fpr", "validation_fpr", "test_fpr", "test_f1", "test_recall"
]])
print("Baseline smoke gate passed.")


## 7. Run or resume all confirmatory-compatible baselines

Expected final output: **480 baseline rows per dataset**. The command is resumable. If Colab disconnects, reconnect and rerun this cell; completed rows are skipped.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/run_confirmatory_baselines.py",
        "--config", "configs/manuscript_readiness.yaml",
        "--profile", "final",
        "--dataset", "all",
        "--confirmatory-results-dir", str(CONFIRMATORY_ROOT),
        "--output-dir", str(MANUSCRIPT_ROOT),
        "--nsl-train", str(NSL_TRAIN),
        "--nsl-test", str(NSL_TEST),
        "--cic-raw-dir", str(CIC_ARCHIVE.parent),
        "--cic-archive", str(CIC_ARCHIVE),
    ],
    cwd=REPO_DIR,
    check=True,
)


## 8. Review baseline completeness before analysis


In [ ]:
import json

for dataset_key in ["nsl_kdd", "cicids2017"]:
    manifest_path = MANUSCRIPT_ROOT / "confirmatory_baselines" / dataset_key / "manifests" / "execution_manifest.json"
    manifest = json.loads(manifest_path.read_text())
    print(dataset_key, manifest["completed_rows"], "/", manifest["expected_rows"])
    assert manifest["completed_rows"] == 480
    assert manifest["test_metrics_used_for_selection"] is False
print("Both baseline executions are complete and compatible.")


## 9. Generate final statistics, tables, and figures


In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/analyze_confirmatory_results.py",
        "--config", "configs/manuscript_readiness.yaml",
        "--profile", "final",
        "--confirmatory-results-dir", str(CONFIRMATORY_ROOT),
        "--baseline-results-dir", str(MANUSCRIPT_ROOT),
        "--output-dir", str(MANUSCRIPT_ROOT / "analysis"),
    ],
    cwd=REPO_DIR,
    check=True,
)


## 10. Run the final manuscript-readiness gate


In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/validate_manuscript_readiness.py",
        "--analysis-dir", str(MANUSCRIPT_ROOT / "analysis"),
    ],
    cwd=REPO_DIR,
    check=True,
)


In [ ]:
audit = pd.read_csv(MANUSCRIPT_ROOT / "analysis" / "publication_tables" / "final_readiness_audit.csv")
display(audit)
assert audit.loc[audit["severity"] == "error", "passed"].astype(bool).all()
print("FINAL MANUSCRIPT READINESS GATE PASSED")


## 11. Review the central paired and calibration evidence

This review is descriptive. Do not manually choose or delete results after seeing the test metrics.


In [ ]:
paired = pd.read_csv(MANUSCRIPT_ROOT / "analysis" / "publication_tables" / "paired_weighted_vs_hamming.csv")
calibration = pd.read_csv(MANUSCRIPT_ROOT / "analysis" / "publication_tables" / "calibration_reliability_summary.csv")
performance = pd.read_csv(MANUSCRIPT_ROOT / "analysis" / "publication_tables" / "confirmatory_performance.csv")

display(paired[paired["metric"].isin(["test_f1", "test_recall", "test_fpr", "test_mcc"])])
display(calibration)
display(performance)


## 12. Build the focused writer handoff package

Because this notebook is stored in Google Drive, Colab autosaves it. Wait for the save indicator to finish before running this cell so the executed notebook can be included.


In [ ]:
command = [
    sys.executable,
    "scripts/build_manuscript_handoff.py",
    "--config", "configs/manuscript_readiness.yaml",
    "--profile", "final",
    "--analysis-dir", str(MANUSCRIPT_ROOT / "analysis"),
    "--confirmatory-results-dir", str(CONFIRMATORY_ROOT),
    "--baseline-results-dir", str(MANUSCRIPT_ROOT),
    "--output-dir", str(HANDOFF_ROOT),
    "--zip",
]
if NOTEBOOK_COPY.exists():
    command.extend(["--executed-notebook", str(NOTEBOOK_COPY)])
else:
    print("Notebook copy was not found at", NOTEBOOK_COPY)
    print("The package will still be created; move this notebook there and rerun this cell to include it.")

subprocess.run(command, cwd=REPO_DIR, check=True)


## 13. Inspect the final package


In [ ]:
import zipfile

archive = Path(str(HANDOFF_ROOT) + ".zip")
if not archive.exists():
    raise FileNotFoundError(archive)
with zipfile.ZipFile(archive) as bundle:
    names = bundle.namelist()
    required = [
        "00_START_HERE.md",
        "03_CONTRIBUTIONS_AND_NOVELTY.md",
        "06_RESULTS_INTERPRETATION.md",
        "08_PROHIBITED_OR_UNSUPPORTED_CLAIMS.md",
        "11_TABLE_AND_FIGURE_MAP.md",
        "PACKAGE_MANIFEST.json",
    ]
    missing = [name for name in required if name not in names]
    if missing:
        raise RuntimeError("Writer package is incomplete: " + str(missing))
    if any(name.startswith("data/") for name in names):
        raise RuntimeError("Raw data must not be present in the writer package.")

print("Final writer package:", archive)
print("Package size (MiB):", round(archive.stat().st_size / (1024 ** 2), 3))
print("Files:", len(names))
print("PROJECT ENGINEERING AND EXPERIMENTAL ANALYSIS COMPLETE")


## Final handoff rule

Send Tanazzah:

1. Access to the private GitHub repository.
2. `FW-LNSA-NIDS/results_manuscript/writer_handoff.zip` or the equivalent `writer_handoff.zip` produced here.
3. The target journal template once Dr. Suraiya confirms the journal.

Do not send raw datasets, smoke outputs, old ZIPs, or superseded exploratory result packages. The repository is the code source of truth; the writer handoff ZIP is the manuscript-evidence source of truth.
